To use Microsoft Agent Framework with Azure OpenAI, you need to install the following Python packages:

In [ ]:
%pip install agent-framework==1.0.0b251223

Note: you may need to restart the kernel to use updated packages.


## Create the agent

First, create a chat client for communicating with Azure OpenAI and use the same login as you used when authenticating with the Azure CLI in the Prerequisites step.
Then, create the agent, providing instructions and a name for the agent.

In [139]:
%pip install agent-framework==1.0.0b251216
# %pip install agent-framework --pre

In [12]:
from agent_framework.azure import AzureOpenAIChatClient

agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]
                             ).create_agent(
    instructions="You are good at telling jokes.",
    name="Joker"
)

result = await agent.run("Tell me a joke about a pirate.")
print(result.text)

Why don’t pirates shower before they walk the plank?  

Because they’ll just wash up on shore! 🏴‍☠️


In [28]:
agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]).create_agent(
    name="VisionAgent",
    instructions="You are a helpful agent that can analyze images"
)

from agent_framework import ChatMessage, TextContent, UriContent, Role

message = ChatMessage(
    role=Role.USER,
    contents=[
        TextContent(text="What do you see in this image?"),
        UriContent(
            uri="https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
            media_type="image/jpeg"
        )
    ]
)

result = await agent.run(message)
print(result.text)

The image depicts a serene landscape featuring a wooden pathway that runs through a lush, green field. The pathway seems to lead into the distance, surrounded by tall grasses and some bushes. Above, the sky is bright with a mix of clouds and blue hues, suggesting a pleasant day. The overall scene conveys a peaceful, natural environment.


In [29]:
thread = agent.get_new_thread()

In [30]:

result1 = await agent.run("Tell me a joke about a pirate.", thread=thread)
print(result1.text)

result2 = await agent.run("Now add some emojis to the joke and tell it in the voice of a pirate's parrot.", thread=thread)
print(result2.text)

Why did the pirate go to school?

Because he wanted to improve his "arrrticulation!"
Arrr, matey! 🦜 Here be the joke, squawk!

Why did the pirate go to school? 🏴‍☠️📚

'Cause he wanted to improve his "arrrrticulation!" 😂🦜

Squawk! Jolly good humor, eh?


In [31]:
for message in await thread.message_store.list_messages():
    print(f"{message.role}: {message.contents[0].text}")

user: Tell me a joke about a pirate.
assistant: Why did the pirate go to school?

Because he wanted to improve his "arrrticulation!"
user: Now add some emojis to the joke and tell it in the voice of a pirate's parrot.
assistant: Arrr, matey! 🦜 Here be the joke, squawk!

Why did the pirate go to school? 🏴‍☠️📚

'Cause he wanted to improve his "arrrrticulation!" 😂🦜

Squawk! Jolly good humor, eh?


In [32]:
thread1 = agent.get_new_thread()
thread2 = agent.get_new_thread()

result1 = await agent.run("Tell me a joke about a pirate.", thread=thread1)
print(result1.text)

result2 = await agent.run("Tell me a joke about a robot.", thread=thread2)
print(result2.text)

result3 = await agent.run("Now add some emojis to the joke and tell it in the voice of a pirate's parrot.", thread=thread1)
print(result3.text)

result4 = await agent.run("Now add some emojis to the joke and tell it in the voice of a robot.", thread=thread2)
print(result4.text)

Why did the pirate go to school?

To improve his "arrrticulation"!
Why did the robot go on a diet?

Because he had too many bytes!
🦜 Ahoy, matey! Here be a joke fer ye! 

Why did the pirate go to school? 🏴‍☠️📚 

To improve his "arrrticulation"! 🦜😂⚓️
🤖 Beep boop! Why did the robot go on a diet? 🥗

Because he had too many bytes! 💻😂


In [22]:
from typing import Annotated
from pydantic import Field

def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [136]:
from typing import Annotated
from pydantic import Field
from agent_framework import ai_function

@ai_function(name="weather_tool", description="Retrieves weather information for any location")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    return f"The weather in {location} is cloudy with a high of 15°C."

In [137]:
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential

agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]).create_agent(
    instructions="You are a helpful assistant",
    tools=get_weather
)

In [138]:
result = await agent.run("What is the weather like in Amsterdam?")
print(result.text)

The weather in Amsterdam is currently cloudy with a high of 15°C.


In [47]:
class WeatherTools:
    def __init__(self):
        self.last_location = None

    @ai_function(name="weather_tool", description="Retrieves weather information for any location")
    def get_weather(
        self,
        location: Annotated[str, Field(description="The location to get the weather for.")],
    ) -> str:
        """Get the weather for a given location."""
        return f"The weather in {location} is cloudy with a high of 15°C."

    @ai_function(name="weather_tool_details", description="Get the detailed weather for the last requested location.")
    def get_weather_details(self) -> int:
        """Get the detailed weather for the last requested location."""
        if self.last_location is None:
            return "No location specified yet."
        return f"The detailed weather in {self.last_location} is cloudy with a high of 15°C, low of 7°C, and 60% humidity."

In [48]:
tools = WeatherTools()
agent = AzureOpenAIChatClient(endpoint=os.environ["AZURE_OPENAI_ENDPOINT"], 
                              deployment_name="gpt-4o-mini", 
                              api_key=os.environ["AZURE_OPENAI_API_KEY"]).create_agent(
    instructions="You are a helpful assistant",
    tools=[tools.get_weather, tools.get_weather_details]
)

## 5. Using function tools with human in the loop approvals

This tutorial step shows you how to use function tools that require human approval with an agent.

When agents require any user input, for example to approve a function call, this is referred to as a human-in-the-loop pattern. An agent run that requires user input, will complete with a response that indicates what input is required from the user, instead of completing with a final answer. The caller of the agent is then responsible for getting the required input from the user, and passing it back to the agent as part of a new agent run.

### Create the agent with function tools requiring approval

When using functions, it's possible to indicate for each function, whether it requires human approval before being executed. This is done by setting the approval_mode parameter to "always_require" when using the @ai_function decorator.

Here is an example of a simple function tool that fakes getting the weather for a given location.

To create a function that requires approval, you can use the approval_mode parameter:



When creating the agent, you can now provide the approval requiring function tool to the agent, by passing a list of tools to the ChatAgent constructor.
Since you now have a function that requires approval, the agent might respond with a request for approval, instead of executing the function directly and returning the result. You can check the response for any user input requests, which indicates that the agent requires user approval for a function.

If there are any function approval requests, the detail of the function call including name and arguments can be found in the function_call property on the user input request. This can be shown to the user, so that they can decide whether to approve or reject the function call.

Once the user has provided their input, you can create a response using the create_response method on the user input request. Pass True to approve the function call, or False to reject it.

The response can then be passed to the agent in a new ChatMessage, to get the result back from the agent.

## 6. Create the agent with structured output

The ChatAgent is built on top of any chat client implementation that supports structured output. The ChatAgent uses the response_format parameter to specify the desired output schema.

When creating or running the agent, you can provide a Pydantic model that defines the structure of the expected output.

Various response formats are supported based on the underlying chat client capabilities.

This example creates an agent that produces structured output in the form of a JSON object that conforms to a Pydantic model schema.

First, define a Pydantic model that represents the structure of the output you want from the agent:

Now you can create an agent using the Azure OpenAI Chat Client:

Now you can run the agent with some textual information and specify the structured output format using the response_format parameter:

The agent response will contain the structured output in the value property, which can be accessed directly as a Pydantic model instance:

## 7. Using an agent as a function tool

This tutorial shows you how to use an agent as a function tool, so that one agent can call another agent as a tool.

You can use a ChatAgent as a function tool by calling .as_tool() on the agent and providing it as a tool to another agent. This allows you to compose agents and build more advanced workflows.

First, create a function tool that will be used by your agent that's exposed as a function.

Create a ChatAgent that uses the function tool.

Now, create a main agent and provide the weather_agent as a function tool by calling .as_tool() to convert weather_agent to a function tool.

Invoke the main agent as normal. It can now call the weather agent as a tool, and should respond in French.

## 8. Expose an agent as an MCP tool

This tutorial shows you how to expose an agent as a tool over the Model Context Protocol (MCP), so it can be used by other systems that support MCP tools.

### Expose an agent as an MCP server

You can expose an agent as an MCP server by using the as_mcp_server() method. This allows the agent to be invoked as a tool by any MCP-compatible client.

First, create an agent that you'll expose as an MCP server. You can also add tools to the agent:

Turn the agent into an MCP server. The agent name and description will be used as the MCP server metadata:

Setup the MCP server to listen for incoming requests over standard input/output:

In [39]:
! python agent_as_mcp_server.py

^C


In [ ]:
{
	"servers": {
		"my-mcp-server": {
			"type": "stdio",
			"command": "python",
			"args": ["D:/projects/ai-course/200_agentic_framework_hello/agent_as_mcp_server.py"],
			"dev": {
				"watch": "**/*.py",
				"debug": {
				"type": "debugpy",
        }
      }
		}
	},
	"inputs": []
}																										

## 9. Enabling observability for Agents

This tutorial shows how to enable OpenTelemetry on an agent so that interactions with the agent are automatically logged and exported. In this tutorial, output is written to the console using the OpenTelemetry console exporter.

### Enable OpenTelemetry in your app

The simplest way to enable observability is using `configure_otel_providers()`.
Two samples are available in the python files `agent_observability_otel_console.py` and `agent_observability_otel_appinsights.py`.
The first sample shows how to configure OpenTelemetry to use the console exporter, while the second sample shows how to configure OpenTelemetry to use the Azure Application Insights exporter.

### 9.1. Running the sample with console exporter

Let's try to run the console exporter sample.
Make sure first to install the dependencies.

In [27]:
! python agent_observability_otel_console.py

d:\projects\ai-course\200_agentic_framework_hello\agent_observability_otel_console.py:35: DeprecationWarning: Use ConsoleLogRecordExporter. Since logs are not stable yet this WILL be removed in future releases.
  exporter = ConsoleLogExporter()
Traceback (most recent call last):
  File "c:\Users\hodellai\AppData\Local\Programs\Python\Python313\Lib\site-packages\agent_framework\observability.py", line 327, in _create_otlp_exporters
    from opentelemetry.exporter.otlp.proto.grpc._log_exporter import OTLPLogExporter as GRPCLogExporter
ModuleNotFoundError: No module named 'opentelemetry.exporter'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "d:\projects\ai-course\200_agentic_framework_hello\agent_observability_otel_console.py", line 106, in <module>
    configure_otel_providers(
    ~~~~~~~~~~~~~~~~~~~~~~~~^
        enable_sensitive_data=True,   # development only
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 

### 9.2. Running the sample with Azure Application Insights exporter

To run the sample with Azure Application Insights exporter, make sure you have the `azure-monitor-opentelemetry` package installed. And also make sure you have an Application Insights resource created in your Azure subscription. Application Insights requires an instrumentation key or Connection String to send telemetry data to the correct resource. You can find the Connection String in the "Overview" section of your Application Insights resource in the Azure portal. Copy the Connection String value.

Now you can run the Application Insights exporter sample. It would be better if you run it from within a Terminal.

You can now view the metrics collected in the Azure portal under your Application Insights resource. It may take a few minutes for the data to appear.

![Application Insights Metrics](./images/observability-app-insights.png)

![Grafana Metrics](./images/observability-grafana.png)

In [35]:
from agent_framework_azure_ai import AzureAIAgentClient
from azure.identity.aio import DefaultAzureCredential

# Using environment variables
# Set AZURE_AI_PROJECT_ENDPOINT=https://your-project.cognitiveservices.azure.com
# Set AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4
credential = DefaultAzureCredential()

# Or passing parameters directly
client = AzureAIAgentClient(
    project_endpoint="https://account-400.services.ai.azure.com/api/projects/project-400", # "https://ai-services-333-400.cognitiveservices.azure.com/",
    model_deployment_name="gpt-4o-mini",
    credential=credential
)

agent = client.create_agent(
        name="GreetingAgent",
        instructions="You are a friendly greeting assistant.",
    )

result = await agent.run("Hello!")

print(result.text)

Hello! How can I assist you today?


In [39]:
from typing import Awaitable
from agent_framework import AgentRunContext
from pyparsing import Callable

async def logging_agent_middleware(
    context: AgentRunContext,
    next: Callable[[AgentRunContext], Awaitable[None]],
) -> None:
    """Simple middleware that logs agent execution."""
    print("Agent starting...")

    # Continue to agent execution
    await next(context)

    print("Agent finished!")

In [ ]:
async def main():
    credential = AzureCliCredential()

    async with AzureAIAgentClient(async_credential=credential).create_agent(
        name="GreetingAgent",
        instructions="You are a friendly greeting assistant.",
        middleware=logging_agent_middleware,  # Add your middleware here
    ) as agent:
        result = await agent.run("Hello!")
        print(result.text)

In [61]:
from agent_framework import FunctionInvocationContext

def get_time():
    """Get the current time."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M:%S")

async def logging_function_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """Middleware that logs function calls."""
    print(f"Calling function: {context.function.name}")

    await next(context)

    print(f"Function result: {context.result}")

# Add both the function and middleware to your agent
client = AzureAIAgentClient(
        credential=credential,
        project_endpoint="https://account-400.services.ai.azure.com/api/projects/project-400",
        model_deployment_name="gpt-4o-mini"
    )

agent = client.create_agent(
    name="TimeAgent",
    instructions="You can tell the current time.",
    tools=[get_time],
    middleware=[logging_function_middleware],
)

result = await agent.run("What time is it?")

await client.close()

Calling function: get_time
Function result: 13:11:05


In [8]:
from azure.identity import AzureCliCredential
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient

agent = ChatAgent(
    chat_client=AzureOpenAIChatClient(
        endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        deployment_name="gpt-4o-mini",
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
    ),
    name="Assistant",
    instructions="You are a helpful assistant.",
)

thread = agent.get_new_thread()

In [9]:
# Run the agent and append the exchange to the thread
response = await agent.run("Tell me a short pirate joke.", thread=thread)
print(response.text)

ServiceResponseException: <class 'agent_framework.azure._chat_client.AzureOpenAIChatClient'> service failed to complete the prompt: Request timed out.

In [ ]:
import json
import os

# Serialize the thread state
serialized_thread = await thread.serialize()
serialized_json = json.dumps(serialized_thread)

# Example: save to a local file (replace with DB or blob storage in production)
file_path = os.path.join(os.getcwd(), "agent_thread.json")
with open(file_path, "w") as f:
    f.write(serialized_json)

In [75]:
# Read persisted JSON
with open(file_path, "r") as f:
    loaded_json = f.read()

reloaded_data = json.loads(loaded_json)

# Deserialize the thread into an AgentThread tied to the same agent type
resumed_thread = await agent.deserialize_thread(reloaded_data)

In [76]:
# Continue the conversation with resumed thread
response = await agent.run("Now tell that joke in the voice of a pirate.", thread=resumed_thread)
print(response.text)

Arrr, matey! Why did th' pirate go t' school? 

T' improve his "arrrticulation"! Har har har!


In [19]:
from dotenv import load_dotenv
import os

if os.path.exists(".env"):
    load_dotenv(override=True)

In [98]:
%pip install redis-entraid redis

Note: you may need to restart the kernel to use updated packages.


In [64]:
from redis import Redis

redis_client = Redis(host=os.environ["AZURE_REDIS_HOST"], 
                     port=os.environ["AZURE_REDIS_PORT"], 
                     password=os.environ["AZURE_REDIS_ACCESS_KEY"], 
                     decode_responses=True,
                     ssl=True)

print("Ping Redis: ", redis_client.ping())

redis_client.set("key01", '{"name":"Adam", "surname":"Doe"}')

redis_client.get("key01")

Ping Redis:  True


'{"name":"Adam", "surname":"Doe"}'

### Using the custom ChatMessageStore with a ChatAgent

To use the custom ChatMessageStore, you need to provide a chat_message_store_factory when creating the agent. This factory allows the agent to create a new instance of the desired ChatMessageStore for each thread.

When creating a ChatAgent, you can provide the chat_message_store_factory parameter in addition to all other agent options.

In [ ]:
from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient

from agent_chat_history_azure_redis import *

redisChatMessageStore = RedisChatMessageStore(
        redis_url=os.environ["AZURE_REDIS_HOST"],
        port=os.environ["AZURE_REDIS_PORT"],
        password=os.environ.get("AZURE_REDIS_ACCESS_KEY"),  # Primary or secondary access key
    )

# Create the chat agent with custom message store factory
agent = ChatAgent(
    chat_client=AzureOpenAIChatClient(
        endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        deployment_name=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
    ),
    name="Joker",
    instructions="You are good at telling jokes.",
    chat_message_store_factory=lambda: redisChatMessageStore,
)

# Use the agent with persistent chat history
thread = agent.get_new_thread()

response = await agent.run("Tell me a joke about pirates", thread=thread)
print(response.text)

Why did the pirate go to school?

To improve his "arrrrrrrrrticulation!"


In [107]:
redis_client.keys("*")

['chat_messages:thread_ac6cf72a-6382-4418-8b8c-49c95d44de04',
 'chat_messages:thread_952b6179-b4d9-4e5b-91ea-f8f5c78a6a15',
 'chat_messages:thread_778e3431-8f99-42b2-aeb2-0f5652e5723c',
 'chat_messages:thread_df89ebeb-ff55-4f29-b82e-276090e5f82f',
 'key01',
 'chat_messages:thread_c3e5fca1-8924-4c70-890f-ff2eed41523f',
 'chat_messages:thread_58455e24-5f74-400e-af2e-fdccc4c5c0ae',
 'chat_messages:thread_cb82327f-925e-4dc0-8b3f-3b55257ac980',
 'chat_messages:thread_131ff16d-178d-40dc-a952-ed15354eab18',
 'chat_messages:thread_437f061a-0c8d-4357-aaa8-47aa380c9c9e',
 'chat_messages:thread_2bff2e4a-59de-4369-8c1b-33c53e7250e2',
 'chat_messages:thread_b001050d-1867-4555-ab54-0826d3a4987b',
 'chat_messages:thread_2c56611d-1b02-440d-ad48-04a558958c96',
 'chat_messages:thread_d5f71cdc-f078-45c3-9839-714f4436ca0b',
 'chat_messages:thread_edac9c06-346b-41ee-a962-87872270e742',
 'chat_messages:thread_9f5671a3-2f5e-485b-9c8a-025ff7d5b4dc',
 'chat_messages:thread_c36d0259-7d8a-4a1d-8e83-9460dda1e5aa'

In [110]:
# redis_client.get("chat_messages:thread_58455e24-5f74-400e-af2e-fdccc4c5c0ae")

for msg in await redisChatMessageStore.list_messages():
    print(f"{msg.role}: {msg.contents[0].text}")

user: Tell me a joke about pirates
assistant: Why did the pirate go to school?

To improve his "arrrrrrrrrticulation!"


In [111]:
for message in await thread.message_store.list_messages():
    print(f"{message.role}: {message.contents[0].text}")

user: Tell me a joke about pirates
assistant: Why did the pirate go to school?

To improve his "arrrrrrrrrticulation!"


In [143]:
from pydantic import BaseModel

class UserInfo(BaseModel):
    name: str | None = None
    age: int | None = None

In [144]:
from collections.abc import MutableSequence, Sequence
from typing import Any
import builtins

from agent_framework import ContextProvider, Context, ChatAgent, ChatClientProtocol, ChatMessage, ChatOptions


class UserInfoMemory(ContextProvider):
    def __init__(self, chat_client: ChatClientProtocol, user_info: UserInfo | None = None, **kwargs: Any):
        """Create the memory.

        If you pass in kwargs, they will be attempted to be used to create a UserInfo object.
        """
        self._chat_client = chat_client
        if user_info:
            self.user_info = user_info
        elif kwargs:
            self.user_info = UserInfo.model_validate(kwargs)
        else:
            self.user_info = UserInfo()

    async def invoked(
        self,
        request_messages: ChatMessage | Sequence[ChatMessage],
        response_messages: ChatMessage | Sequence[ChatMessage] | None = None,
        invoke_exception: Exception | None = None,
        **kwargs: Any,
    ) -> None:
        """Extract user information from messages after each agent call."""
        # Ensure request_messages is a list
        messages_list = (
            [request_messages]
            if isinstance(request_messages, ChatMessage)
            else builtins.list(request_messages)
        )
        # messages_list = [request_messages] if isinstance(request_messages, ChatMessage) else list(request_messages)

        # Check if we need to extract user info from user messages
        user_messages = [msg for msg in messages_list if msg.role.value == "user"]

        if (self.user_info.name is None or self.user_info.age is None) and user_messages:
            try:
                # Use the chat client to extract structured information
                result = await self._chat_client.get_response(
                    messages=messages_list,
                    chat_options=ChatOptions(
                        instructions=(
                            "Extract the user's name and age from the message if present. "
                            "If not present return nulls."
                        ),
                        response_format=UserInfo,
                    ),
                )

                # Update user info with extracted data
                if result.value and isinstance(result.value, UserInfo):
                    if self.user_info.name is None and result.value.name:
                        self.user_info.name = result.value.name
                    if self.user_info.age is None and result.value.age:
                        self.user_info.age = result.value.age

            except Exception:
                pass  # Failed to extract, continue without updating

    async def invoking(self, messages: ChatMessage | MutableSequence[ChatMessage], **kwargs: Any) -> Context:
        """Provide user information context before each agent call."""
        instructions: list[str] = []

        if self.user_info.name is None:
            instructions.append(
                "Ask the user for their name and politely decline to answer any questions until they provide it."
            )
        else:
            instructions.append(f"The user's name is {self.user_info.name}.")

        if self.user_info.age is None:
            instructions.append(
                "Ask the user for their age and politely decline to answer any questions until they provide it."
            )
        else:
            instructions.append(f"The user's age is {self.user_info.age}.")

        # Return context with additional instructions
        return Context(instructions=" ".join(instructions))

    def serialize(self) -> str:
        """Serialize the user info for thread persistence."""
        return self.user_info.model_dump_json()

In [ ]:
import asyncio
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity import AzureCliCredential, DefaultAzureCredential
# from azure.identity.aio import AzureCliCredential

credential = DefaultAzureCredential()
# credential = AzureCliCredential()

chat_client = AzureAIAgentClient(
    credential=credential,
    project_endpoint="https://account-400.services.ai.azure.com/api/projects/project-400",  # "https://ai-services-333-400.cognitiveservices.azure.com/",
    model_deployment_name="gpt-4o-mini",
)

# Create the memory provider
memory_provider = UserInfoMemory(chat_client)

# Create the agent with memory
agent = ChatAgent(
    chat_client=chat_client,
    instructions="You are a friendly assistant. Always address the user by their name.",
    context_providers=memory_provider,
)

# Create a new thread for the conversation
thread = agent.get_new_thread()

print(await agent.run("Hello, what is the square root of 9?", thread=thread))

print(await agent.run("My name is Ruaidhrí", thread=thread))

print(await agent.run("I am 20 years old", thread=thread))

# Access the memory component via the thread's context_providers attribute and inspect the memories
if thread.context_provider:
    user_info_memory = thread.context_provider.providers[0]
    if isinstance(user_info_memory, UserInfoMemory):
        print()
        print(f"MEMORY - User Name: {user_info_memory.user_info.name}")
        print(f"MEMORY - User Age: {user_info_memory.user_info.age}")

Hello! Before I can help you with that, could you please tell me your name?
Thank you, Ruaidhrí! It's nice to meet you. Could you please tell me your age?
Great, thank you for sharing that, Ruaidhrí! Now, how can I assist you today?

MEMORY - User Name: Ruaidhrí
MEMORY - User Age: 20


The source code is also available in the file `agent_memory.py`.